In [51]:
import pypsa

In [52]:
n = pypsa.Network('plots/strategy_model_solved.nc')

batt_p_nom_w = 6000.0  # 6 kW
batt_max_hours = 2.5
batt_max_hours_quarterly = 2.5 / 0.25 # 15 min intervals
batt_capacity_wh = batt_p_nom_w * batt_max_hours

INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, loads, storage_units


In [53]:
n.storage_units['state_of_charge_initial']

StorageUnit
battery    58800.0
Name: state_of_charge_initial, dtype: float64

In [54]:
n.optimize()

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.1s
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 485 primals, 1164 duals
Objective: 1.65e+01
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


Running HiGHS 1.11.0 (git hash: n/a): Copyright (c) 2025 HiGHS under MIT licence terms
LP   linopy-problem-6qr_qv3s has 1164 rows; 485 cols; 1745 nonzeros
Coefficient ranges:
  Matrix [9e-01, 1e+00]
  Cost   [2e-04, 5e-04]
  Bound  [0e+00, 0e+00]
  RHS    [5e-13, 6e+04]
Presolving model
194 rows, 466 cols, 756 nonzeros  0s
Dependent equations search running on 194 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
194 rows, 466 cols, 756 nonzeros  0s
Presolve : Reductions: rows 194(-970); columns 466(-19); elements 756(-989)
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Pr: 98(207041) 0s
        317     1.6511343188e+01 Pr: 0(0) 0s
Solving the original LP from the solution after postsolve
Model name          : linopy-problem-6qr_qv3s
Model status        : Optimal
Simplex   iterations: 317
Objective value    

('ok', 'optimal')

In [ ]:
n.optimize.create_model()

ctx = {}
ctx['soc_limit_lower'] = 20  # in percent
ctx['soc_limit_upper'] = 80  # in percent

n.model.add_constraints(
    n.model.variables['StorageUnit-state_of_charge'],
    ">=",
    (ctx['soc_limit_lower'] / 100.0) * batt_capacity_wh_pypsa,
    'battery_soc_min'
)
n.model.add_constraints(
    n.model.variables['StorageUnit-state_of_charge'],
    "<=",
    (ctx['soc_limit_upper'] / 100.0) * batt_capacity_wh_pypsa,
    'battery_soc_max'
)

n.optimize.solve_model()

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.13s
Status: warning
Termination condition: infeasible
Solution: 0 primals, 0 duals
Objective: nan
Solver model: available
Solver message: Infeasible



Running HiGHS 1.11.0 (git hash: n/a): Copyright (c) 2025 HiGHS under MIT licence terms
LP   linopy-problem-si1skrxf has 1358 rows; 485 cols; 1939 nonzeros
Coefficient ranges:
  Matrix [9e-01, 1e+00]
  Cost   [2e-04, 5e-04]
  Bound  [0e+00, 0e+00]
  RHS    [5e-13, 6e+04]
Presolving model
194 rows, 466 cols, 756 nonzeros  0s
Problem status detected on presolve: Infeasible
Model name          : linopy-problem-si1skrxf
Model status        : Infeasible
Objective value     :  0.0000000000e+00
HiGHS run time      :          0.00
Writing the solution to /tmp/linopy-solve-yff74cvh.sol


('warning', 'infeasible')

In [56]:
batt_capacity_wh*1

15000.0